# Protein Backbone Diffusion V4

V4 starts from the stable V3b geometry-loss pipeline and replaces the flattened denoiser with a pragmatic EGNN-style coordinate-aware graph model.
The data splits, chunking, masking, train-only coordinate normalization, 100-step linear noise schedule, geometry-augmented DDPM objective, checkpointing, sampling, and structural diagnostics stay aligned with V3b so the architectural change is isolated and reportable.


## 1. Runtime and setup

This section detects the runtime, optionally mounts Google Drive, locates or clones the repository, resolves the CATH data directory, installs small notebook dependencies, and sets the seed/device/output directories. By default V4 writes artifacts to Google Drive when Drive mounts successfully, otherwise it falls back to the repo-local `results/v4/` directory. Google Drive is no longer required for PyCharm or other IDE execution.

Runtime flags for local or IDE execution:

- set `BACKBONE_DIFFUSION_MOUNT_DRIVE=0` to skip Google Drive mounting; this also disables table writes by default
- set `BACKBONE_DIFFUSION_SAVE_TABLES=0` to disable CSV/Parquet table artifact writes explicitly
- set `BACKBONE_DIFFUSION_SAVE_TABLES=1` to write tables even when Drive mounting is skipped
- set `BACKBONE_DIFFUSION_ARTIFACT_BASE_DIR=/path/to/results` to override the artifact root


In [ ]:
import base64
import os
import random
import subprocess
import sys
import time
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch.utils.data import DataLoader

try:
    import google.colab  # type: ignore
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None  # type: ignore
    userdata = None  # type: ignore
    IN_COLAB = False

def env_flag(name: str, default: bool) -> bool:
    """Parse a boolean environment flag."""
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

DRIVE_MOUNT_MODE = os.environ.get('BACKBONE_DIFFUSION_MOUNT_DRIVE', 'auto').strip().lower()
DRIVE_MOUNT_SKIPPED = DRIVE_MOUNT_MODE in {'0', 'false', 'no', 'off', 'skip'}
SAVE_TABLE_ARTIFACTS = env_flag('BACKBONE_DIFFUSION_SAVE_TABLES', not DRIVE_MOUNT_SKIPPED)

DRIVE_MOUNTPOINT = Path('/content/drive')
DRIVE_MYDRIVE = DRIVE_MOUNTPOINT / 'MyDrive'
DRIVE_MOUNTED = False

if IN_COLAB and drive is not None and not DRIVE_MOUNT_SKIPPED:
    try:
        drive.mount(str(DRIVE_MOUNTPOINT), force_remount=False)
        DRIVE_MOUNTED = DRIVE_MYDRIVE.exists()
    except Exception as exc:  # noqa: BLE001
        if DRIVE_MOUNT_MODE in {'1', 'true', 'yes', 'on', 'required'}:
            raise RuntimeError('Google Drive mounting was explicitly requested but failed.') from exc
        print(f'Google Drive mount skipped after failure: {exc}')
else:
    print('Google Drive mount skipped.')

if DRIVE_MOUNTED:
    print(f'Drive mounted: {DRIVE_MYDRIVE}')
else:
    print('Running without Google Drive mount; artifacts default to the repo-local results directory.')

REPO_REMOTE_URL = os.environ.get('GITHUB_REPO_URL', 'https://github.com/mitsenkov/latent-structure-diffusion.git')
REPO_BRANCH = os.environ.get('GITHUB_REPO_BRANCH', 'main')
REPO_CLONE_DIR = Path(os.environ.get('REPO_CLONE_DIR', '/content/latent-structure-diffusion'))
REPO_DRIVE_DIR = Path(os.environ.get('REPO_DRIVE_DIR', '/content/drive/MyDrive/latent-structure-diffusion'))
DEFAULT_CATH_FOLDER_ID = os.environ.get('CATH_SHARED_FOLDER_ID', '')
LOCAL_DATA_CACHE = Path(os.environ.get('CATH_LOCAL_CACHE', '/content/cath_backbone_data'))
ALLOW_GIT_CLONE = os.environ.get('ALLOW_GIT_CLONE', '1' if IN_COLAB else '0') == '1'

def get_github_token() -> str | None:
    """Return a GitHub token from env, Colab userdata, or an interactive prompt."""
    token = os.environ.get('GITHUB_TOKEN')
    if token:
        return token.strip()
    if IN_COLAB:
        try:
            token = userdata.get('GITHUB_TOKEN')
        except Exception:  # noqa: BLE001
            token = None
        if token:
            return str(token).strip()
    if ALLOW_GIT_CLONE:
        token = getpass('Paste the GitHub token with read access to this repo: ').strip()
        if token:
            return token
    return None

def find_repo_root(start_paths: list[Path] | None = None) -> Path | None:
    """Return the repository root if a checkout is already present."""
    candidates = start_paths or [Path.cwd().resolve(), REPO_CLONE_DIR, REPO_DRIVE_DIR]
    for start in candidates:
        if not start.exists():
            continue
        for candidate in [start, *start.parents]:
            if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
                return candidate
    return None

def build_auth_header(token: str) -> str:
    """Build a Git basic auth header for private GitHub clone access."""
    raw = f'x-access-token:{token}'.encode('utf-8')
    encoded = base64.b64encode(raw).decode('ascii')
    return f'AUTHORIZATION: basic {encoded}'

def bootstrap_repo() -> Path:
    """Make the repo available in Colab or reuse an existing checkout."""
    existing_root = find_repo_root()
    if existing_root is not None:
      if IN_COLAB:
          token = get_github_token()
          if token:
              pull_cmd = [
                  'git',
                  '-C',
                  str(existing_root),
                  '-c',
                  f'http.extraheader={build_auth_header(token)}',
                  'pull',
                  '--ff-only',
              ]
              subprocess.run(pull_cmd, check=True, capture_output=True, text=True)
      return existing_root

    if not IN_COLAB:
        raise FileNotFoundError(
            'Could not locate the repository root. Expected a folder containing pyproject.toml and src/.'
        )

    if not ALLOW_GIT_CLONE:
        raise FileNotFoundError(
            'Could not locate a local repo checkout. In Colab, set ALLOW_GIT_CLONE=1 or define GITHUB_TOKEN.'
        )

    token = get_github_token()
    if not token:
        raise FileNotFoundError(
            'GitHub cloning was enabled, but no token was provided.'
        )

    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    clone_cmd = [
        'git',
        '-c',
        f'http.extraheader={build_auth_header(token)}',
        'clone',
        '--branch',
        REPO_BRANCH,
        REPO_REMOTE_URL,
        str(REPO_CLONE_DIR),
    ]
    try:
        subprocess.run(clone_cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as exc:
        stderr = (exc.stderr or '').strip()
        stdout = (exc.stdout or '').strip()
        details = '\n'.join(part for part in [stdout, stderr] if part)
        raise RuntimeError(
            'GitHub clone failed in this Colab runtime. The token may be missing, invalid, or lack repo read access. '            f'Command: git clone --branch {REPO_BRANCH} {REPO_REMOTE_URL} {REPO_CLONE_DIR}\n{details}'
        ) from exc
    return REPO_CLONE_DIR

def ensure_gdown_installed() -> None:
    """Install gdown on demand."""
    try:
        import gdown  # type: ignore  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])

def resolve_data_dir() -> Path:
    """Resolve the dataset directory without requiring a Drive mount."""
    env_dir = os.environ.get('CATH_DATA_DIR')
    if env_dir:
        candidate = Path(env_dir)
        if (candidate / 'chain_set.jsonl').exists() and (candidate / 'chain_set_splits.json').exists():
            return candidate

    local_candidate = LOCAL_DATA_CACHE
    if (local_candidate / 'chain_set.jsonl').exists() and (local_candidate / 'chain_set_splits.json').exists():
        return local_candidate

    if IN_COLAB:
        ensure_gdown_installed()
        import gdown  # type: ignore

        local_candidate.mkdir(parents=True, exist_ok=True)
        url = f'https://drive.google.com/drive/folders/{DEFAULT_CATH_FOLDER_ID}'
        gdown.download_folder(url=url, output=str(local_candidate), quiet=False, use_cookies=False)
        if (local_candidate / 'chain_set.jsonl').exists() and (local_candidate / 'chain_set_splits.json').exists():
            return local_candidate

    raise FileNotFoundError(
        'Could not resolve the CATH dataset directory. Set CATH_DATA_DIR to a local path with chain_set.jsonl '
        'and chain_set_splits.json, or allow the notebook to download the public shared folder.'
    )

REPO_ROOT = bootstrap_repo()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = resolve_data_dir()

DEFAULT_ARTIFACT_DIR = (
    DRIVE_MYDRIVE / 'latent-structure-generation' / 'results'
    if DRIVE_MOUNTED
    else REPO_ROOT / 'results'
)
ARTIFACT_BASE_DIR = Path(os.environ.get('BACKBONE_DIFFUSION_ARTIFACT_BASE_DIR', str(DEFAULT_ARTIFACT_DIR)))
ARTIFACT_RUN_NAME = os.environ.get('BACKBONE_DIFFUSION_RUN_NAME', 'v4')
ARTIFACT_DIR = ARTIFACT_BASE_DIR / ARTIFACT_RUN_NAME
FIGURE_DIR = ARTIFACT_DIR / 'figures'
TABLE_DIR = ARTIFACT_DIR / 'tables'
CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
for directory in [FIGURE_DIR, TABLE_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def save_table_artifact(frame: pd.DataFrame, name: str, *, drop_columns: list[str] | None = None) -> dict[str, Path]:
    """Save a dataframe as CSV and Parquet under the run's tables directory."""
    if not SAVE_TABLE_ARTIFACTS:
        return {}
    export_frame = frame.drop(columns=[column for column in (drop_columns or []) if column in frame.columns]).copy()
    csv_path = TABLE_DIR / f'{name}.csv'
    parquet_path = TABLE_DIR / f'{name}.parquet'
    export_frame.to_csv(csv_path, index=False)
    try:
        export_frame.to_parquet(parquet_path, index=False)
    except Exception as exc:  # noqa: BLE001
        print(f'Could not write Parquet for {name}: {exc}')
    return {'csv': csv_path, 'parquet': parquet_path}

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(0)
else:
    device_name = 'CPU'

try:
    import py3Dmol  # type: ignore
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'py3Dmol'])
    import py3Dmol  # type: ignore

if SAVE_TABLE_ARTIFACTS:
    try:
        import pyarrow  # type: ignore  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyarrow'])

from latent_structure_generation.backbone_diffusion import (
    BACKBONE_ATOMS,
    BackboneDataset,
    BackboneCoordinateEGNNDenoiser,
    BackboneNormalizationStats,
    apply_coordinate_normalisation,
    backbone_coords_to_protein,
    backbone_structure_summary,
    build_normalization_stats_from_dataframe,
    build_overlapping_backbone_chunks,
    collate_backbone_examples,
    centre_coordinates,
    create_noise_schedule,
    extract_backbone_from_coords_dict,
    flatten_backbone,
    invert_coordinate_normalisation,
    masked_coordinate_rmse,
    masked_noise_mse,
    pad_or_crop_backbone,
    predict_x0,
    q_sample,
    sample_backbone,
    sample_timesteps,
    structure_validity_report,
    to_pdb,
    unflatten_backbone,
)
from latent_structure_generation.plots import plot_ca_trace

print(f'Repository root: {REPO_ROOT}')
print(f'Data directory: {DATA_DIR}')
print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Google Drive mounted: {DRIVE_MOUNTED}')
print(f'Table artifact saving enabled: {SAVE_TABLE_ARTIFACTS}')
print(f'Seed: {SEED}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Selected device: {device}')
print(f'Device name: {device_name}')
print(f'GitHub clone enabled: {ALLOW_GIT_CLONE}')
print(f'GitHub token present: {bool(os.environ.get("GITHUB_TOKEN") or (IN_COLAB and "GITHUB_TOKEN" in getattr(userdata, "keys", lambda: [])()))}')



## 2. Load the starter CATH tables

This reuses the provided Google Drive layout from the starter notebook. The split names are canonicalized to `train`, `validation`, and `test` so later code does not depend on `val` vs `validation` naming drift.


In [ ]:
chain_set_path = DATA_DIR / 'chain_set.jsonl'
split_path = DATA_DIR / 'chain_set_splits.json'

print(f'Reading {chain_set_path}')
df = pd.read_json(chain_set_path, lines=True)
print(f'Reading {split_path}')
chain_splits = pd.read_json(split_path, lines=True)

def canonical_split_name(name: str) -> str:
    """Map split labels to the canonical notebook names."""
    return {'val': 'validation', 'validation': 'validation', 'train': 'train', 'test': 'test'}.get(name, name)

split_lookup: dict[str, str] = {}

# Preserve the true train/validation/test buckets first.
for column in ['train', 'validation', 'test']:
    if column not in chain_splits.columns:
        continue
    canonical = canonical_split_name(column)
    values = chain_splits[column].iloc[0]
    for record_name in values:
        split_lookup.setdefault(record_name, canonical)

# cath_nodes is auxiliary metadata; use it only for records not already assigned.
if 'cath_nodes' in chain_splits.columns:
    cath_nodes = chain_splits['cath_nodes'].iloc[0]
    for record_name in cath_nodes.keys():
        split_lookup.setdefault(record_name, 'cath_nodes')

df['split'] = df['name'].map(split_lookup).fillna('unknown')
df['split'] = df['split'].map(canonical_split_name)

available_splits = sorted(df['split'].dropna().unique().tolist())
print('Available splits:', available_splits)
print('Split counts:')
print(df['split'].value_counts(dropna=False).sort_index())
print('Columns:', list(df.columns))
print('Example row keys:', list(df.iloc[0].index))

save_table_artifact(df, 'raw_chain_manifest', drop_columns=['coords'])


## 3. Data audit before training

This is the required review phase. It checks shapes, masks, invalid values, real lengths, truncation, coordinate ranges, backbone bond lengths, and a few visual examples before any training code runs.


In [ ]:
from collections import defaultdict

def move_batch_to_device(batch: dict[str, object], target_device: torch.device) -> dict[str, object]:
    """Move tensor values in a batch to the selected device."""
    moved: dict[str, object] = {}
    for key, value in batch.items():
        moved[key] = value.to(target_device) if torch.is_tensor(value) else value
    return moved

def audit_dataframe(split_df: pd.DataFrame, split_name: str, max_length: int) -> pd.DataFrame:
    """Audit one split and return a per-example summary table."""
    rows: list[dict[str, object]] = []
    for _, row in split_df.reset_index(drop=True).iterrows():
        record_id = row['name']
        try:
            coords, residue_mask = extract_backbone_from_coords_dict(row['coords'])
            raw_length = int(coords.shape[0])
            real_length = int(residue_mask.sum())
            coords_fixed, mask_fixed, truncated = pad_or_crop_backbone(coords, residue_mask, max_length)
            coords_t = torch.tensor(coords_fixed, dtype=torch.float32)
            mask_t = torch.tensor(mask_fixed.astype(np.float32), dtype=torch.float32)
            has_nan = bool(np.isnan(coords_fixed).any())
            has_inf = bool(np.isinf(coords_fixed).any())
            zero_real_residues = int(((np.abs(coords_fixed).sum(axis=(1, 2)) == 0.0) & mask_fixed).sum())
            summary = backbone_structure_summary(coords_t, mask_t)
            rows.append(
                {
                    'record_id': record_id,
                    'split': split_name,
                    'raw_length': raw_length,
                    'real_length': real_length,
                    'padded_length': int(mask_fixed.shape[0]),
                    'truncated': bool(truncated),
                    'has_nan': has_nan,
                    'has_inf': has_inf,
                    'zero_real_residues': zero_real_residues,
                    'keep_example': bool(real_length > 0 and not has_nan and not has_inf),
                    **summary,
                }
            )
        except Exception as exc:  # noqa: BLE001
            rows.append(
                {
                    'record_id': record_id,
                    'split': split_name,
                    'raw_length': np.nan,
                    'real_length': 0,
                    'padded_length': max_length,
                    'truncated': False,
                    'has_nan': True,
                    'has_inf': True,
                    'zero_real_residues': 0,
                    'keep_example': False,
                    'audit_error': str(exc),
                    'n_residues': 0,
                    'mean_adjacent_ca': np.nan,
                    'fraction_adjacent_ca_in_band': np.nan,
                    'mean_n_ca': np.nan,
                    'mean_ca_c': np.nan,
                    'mean_c_o': np.nan,
                    'mean_c_n': np.nan,
                    'radius_of_gyration': np.nan,
                }
            )
    return pd.DataFrame(rows)

def summarize_lengths(audit_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize real lengths, padding, and truncation for one split."""
    if audit_df.empty:
        return pd.DataFrame([
            {
                'split': 'unknown',
                'n_examples': 0,
                'min_real_length': np.nan,
                'median_real_length': np.nan,
                'max_real_length': np.nan,
                'mean_real_length': np.nan,
                'mean_padding_fraction': np.nan,
                'truncated_fraction': np.nan,
            }
        ])

    if 'keep_example' not in audit_df.columns:
        valid = audit_df.copy()
    else:
        valid = audit_df[audit_df['keep_example'].fillna(False)].copy()
    if valid.empty:
        split_name = audit_df['split'].iloc[0] if 'split' in audit_df.columns and len(audit_df) else 'unknown'
        return pd.DataFrame([
            {
                'split': split_name,
                'n_examples': 0,
                'min_real_length': np.nan,
                'median_real_length': np.nan,
                'max_real_length': np.nan,
                'mean_real_length': np.nan,
                'mean_padding_fraction': np.nan,
                'truncated_fraction': np.nan,
            }
        ])

    lengths = valid['real_length'].astype(int)
    padded_fraction = 1.0 - (lengths / valid['padded_length'].astype(int))
    return pd.DataFrame(
        [
            {
                'split': valid['split'].iloc[0],
                'n_examples': int(len(valid)),
                'min_real_length': int(lengths.min()),
                'median_real_length': float(lengths.median()),
                'max_real_length': int(lengths.max()),
                'mean_real_length': float(lengths.mean()),
                'mean_padding_fraction': float(padded_fraction.mean()),
                'truncated_fraction': float(valid['truncated'].mean()),
            }
        ]
    )

def coordinate_moments(split_df: pd.DataFrame, max_length: int) -> dict[str, np.ndarray | float]:
    """Compute mean/std/RMS statistics over real atom coordinates only."""
    sum_xyz = np.zeros(3, dtype=np.float64)
    sum_sq_xyz = np.zeros(3, dtype=np.float64)
    count = 0
    rg_values: list[float] = []
    for _, row in split_df.reset_index(drop=True).iterrows():
        coords, residue_mask = extract_backbone_from_coords_dict(row['coords'])
        if residue_mask.sum() == 0:
            continue
        coords_fixed, mask_fixed, _ = pad_or_crop_backbone(coords, residue_mask, max_length)
        coords_t = torch.tensor(coords_fixed, dtype=torch.float32)
        mask_t = torch.tensor(mask_fixed.astype(np.float32), dtype=torch.float32)
        centred = centre_coordinates(coords_t.unsqueeze(0), mask_t.unsqueeze(0))[0]
        real = centred[mask_t.bool()].reshape(-1, 3)
        sum_xyz += real.sum(dim=0).cpu().numpy()
        sum_sq_xyz += (real.pow(2)).sum(dim=0).cpu().numpy()
        count += int(real.shape[0])
        rg_values.append(float(backbone_structure_summary(coords_t, mask_t)['radius_of_gyration']))
    mean = sum_xyz / max(count, 1)
    var = sum_sq_xyz / max(count, 1) - mean**2
    std = np.sqrt(np.clip(var, 1e-6, None))
    return {'mean': mean, 'std': std, 'rms': float(np.sqrt(np.mean(sum_sq_xyz / max(count, 1)))), 'count': count, 'radius_of_gyration_mean': float(np.mean(rg_values)) if rg_values else float('nan')}

MAX_SEQ_LENGTH = 256
train_audit = audit_dataframe(df[df['split'] == 'train'], 'train', MAX_SEQ_LENGTH)
val_audit = audit_dataframe(df[df['split'] == 'validation'], 'validation', MAX_SEQ_LENGTH)
test_audit = audit_dataframe(df[df['split'] == 'test'], 'test', MAX_SEQ_LENGTH)

audit_df = pd.concat([train_audit, val_audit, test_audit], ignore_index=True)

length_summary_df = pd.concat(
    [summarize_lengths(train_audit), summarize_lengths(val_audit), summarize_lengths(test_audit)],
    ignore_index=True,
)

print('Audit summary by split:')
display(length_summary_df)

print('Invalid / filtered examples by split:')
filter_summary = (
    audit_df.groupby('split', as_index=False)
    .agg(total_examples=('record_id', 'count'), kept_examples=('keep_example', 'sum'), invalid_examples=('keep_example', lambda s: int((~s).sum())), truncated_examples=('truncated', 'sum'))
)
filter_summary['filtered_out'] = filter_summary['total_examples'] - filter_summary['kept_examples']
display(filter_summary)

print('Coordinate and structure summary on the training split:')
train_moments = coordinate_moments(df[df['split'] == 'train'], MAX_SEQ_LENGTH)
print(train_moments)

print('Head of the audit table:')
display(audit_df.head(8))

save_table_artifact(train_audit, 'train_audit')
save_table_artifact(val_audit, 'validation_audit')
save_table_artifact(test_audit, 'test_audit')
save_table_artifact(audit_df, 'audit_table')
save_table_artifact(length_summary_df, 'length_summary')
save_table_artifact(filter_summary, 'filter_summary')

print('Shape checks:')
print('coords expected shape: (B, L, 4, 3)')
print('mask expected shape: (B, L)')
print('backbone atom axis:', BACKBONE_ATOMS)
print('training split examples kept:', int(train_audit['keep_example'].sum()))
print('validation split examples kept:', int(val_audit['keep_example'].sum()))
print('test split examples kept:', int(test_audit['keep_example'].sum()))


### What the audit established

The audit keeps padded positions, drops only examples with no valid backbone residues, and reports the amount of truncation caused by fixed-length batching. The same summary tables also give the train-only statistics needed for normalization. Long training chains are then expanded into overlapping fixed-length chunks so we stop throwing away so much of each protein.


In [ ]:
# Build chain-level filtered split tables first.
train_chain_df = df[df['split'] == 'train'].reset_index(drop=True)
validation_chain_df = df[df['split'] == 'validation'].reset_index(drop=True)
test_chain_df = df[df['split'] == 'test'].reset_index(drop=True)

train_df = train_chain_df[train_audit['keep_example'].values].reset_index(drop=True)
validation_df = validation_chain_df[val_audit['keep_example'].values].reset_index(drop=True)
test_df = test_chain_df[test_audit['keep_example'].values].reset_index(drop=True)

# Expand long training chains into overlapping 256-residue windows.
TRAIN_CHUNK_STRIDE = 128
train_chunked_df = build_overlapping_backbone_chunks(
    train_df,
    split='train',
    max_length=MAX_SEQ_LENGTH,
    stride=TRAIN_CHUNK_STRIDE,
    record_id_column='name',
)

print('Filtered split sizes:')
print('train:', len(train_df))
print('validation:', len(validation_df))
print('test:', len(test_df))
print('train chunks:', len(train_chunked_df))
print('unique train parent chains:', train_chunked_df['parent_chain_id'].nunique())
print('training chunk stride:', TRAIN_CHUNK_STRIDE)
chunk_count_summary = train_chunked_df.groupby('parent_chain_id').size()
print('chunk count per parent chain: min', int(chunk_count_summary.min()), 'median', float(chunk_count_summary.median()), 'max', int(chunk_count_summary.max()))
print('chunk length range:', int(train_chunked_df['chunk_length'].min()), 'to', int(train_chunked_df['chunk_length'].max()))
chunk_count_summary_df = chunk_count_summary.rename_axis('parent_chain_id').reset_index(name='chunk_count')

save_table_artifact(train_df, 'train_chains_manifest', drop_columns=['coords'])
save_table_artifact(validation_df, 'validation_chains_manifest', drop_columns=['coords'])
save_table_artifact(test_df, 'test_chains_manifest', drop_columns=['coords'])
save_table_artifact(train_chunked_df, 'train_chunked_manifest', drop_columns=['coords'])
save_table_artifact(chunk_count_summary_df, 'train_chunk_counts')

train_dataset = BackboneDataset(train_chunked_df, split='train', max_length=MAX_SEQ_LENGTH, record_id_column='chunk_id')
validation_dataset = BackboneDataset(validation_df, split='validation', max_length=MAX_SEQ_LENGTH)
test_dataset = BackboneDataset(test_df, split='test', max_length=MAX_SEQ_LENGTH)

BATCH_SIZE = 32
NUM_WORKERS = 2 if IN_COLAB else 0
PIN_MEMORY = device.type == 'cuda'

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)

example_batch = next(iter(train_loader))
print('Example batch keys:', list(example_batch.keys()))
print('coords shape:', tuple(example_batch['coords'].shape), 'dtype:', example_batch['coords'].dtype, 'device:', example_batch['coords'].device)
print('mask shape:', tuple(example_batch['mask'].shape), 'dtype:', example_batch['mask'].dtype, 'device:', example_batch['mask'].device)
print('lengths shape:', tuple(example_batch['length'].shape))
print('real_length shape:', tuple(example_batch['real_length'].shape))
print('truncated shape:', tuple(example_batch['truncated'].shape))
print('first record ids:', example_batch['record_id'][:3])
print('first parent chain ids:', example_batch['parent_chain_id'][:3])
print('first chunk ids:', example_batch['chunk_id'][:3])


## 4. Visual sanity checks on real data

This shows a few training examples before any normalization or diffusion is applied. The point is to confirm that the backbone traces and masks line up with the expected protein geometry.


In [ ]:
def plot_ca_trace_from_batch(coords: torch.Tensor, mask: torch.Tensor, title: str):
    """Plot a CA trace using the existing plotting helper."""
    ca = coords[mask.bool(), 1, :].detach().cpu().numpy()
    fig, ax = plot_ca_trace(ca, title=title)
    plt.show()
    return fig, ax

def render_backbone_example(coords: torch.Tensor, mask: torch.Tensor, title: str):
    """Render one backbone structure as a PDB-backed py3Dmol viewer."""
    protein = backbone_coords_to_protein(coords, mask)
    pdb_str = to_pdb(protein)
    view = py3Dmol.view(width=700, height=500)
    view.addModel(pdb_str, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    view.show()
    return view

for index in range(min(2, example_batch['coords'].shape[0])):
    real_coords = example_batch['coords'][index]
    real_mask = example_batch['mask'][index]
    print(f'Real example {index}:', example_batch['record_id'][index])
    print(backbone_structure_summary(real_coords, real_mask))
    plot_ca_trace_from_batch(real_coords, real_mask.bool(), title=f"Real CA trace: {example_batch['record_id'][index]}")
    render_backbone_example(real_coords, real_mask, title=f"Real backbone: {example_batch['record_id'][index]}")


## 5. Preprocessing and train-only normalization

The model sees centered backbone coordinates that are standardized with statistics computed from the training split only. This cell also checks the flatten / unflatten / inverse-transform path before training starts.


In [ ]:
normalization_stats = build_normalization_stats_from_dataframe(train_df)
print('Normalization mean:', normalization_stats.mean)
print('Normalization std:', normalization_stats.std)

smoke_batch = next(iter(train_loader))
coords = smoke_batch['coords']
mask = smoke_batch['mask']
coords_centered = centre_coordinates(coords, mask)
coords_norm = apply_coordinate_normalisation(coords_centered, mask, normalization_stats)
coords_flat = flatten_backbone(coords_norm)
coords_roundtrip = unflatten_backbone(coords_flat)
coords_denorm = invert_coordinate_normalisation(coords_roundtrip, normalization_stats)

print('coords_centered:', tuple(coords_centered.shape))
print('coords_norm:', tuple(coords_norm.shape))
print('coords_flat:', tuple(coords_flat.shape))
print('coords_roundtrip:', tuple(coords_roundtrip.shape))
print('coords_denorm:', tuple(coords_denorm.shape))
print('max inverse diff:', float((coords_denorm - coords_centered).abs().max().item()))
print('smoke-test mask sum:', float(mask.sum().item()))


## 6. Diffusion utilities and model

V4 keeps the V3b geometry-augmented DDPM objective but changes the denoiser architecture. The new model treats each `N`, `CA`, `C`, and `O` atom as a graph node with atom-type identity, residue-position conditioning, timestep conditioning, and sparse backbone edges. EGNN-style message passing uses squared distances for hidden-state updates and relative-coordinate updates for equivariant coordinate refinement.


In [ ]:
TIMESTEPS = 100
noise_schedule = create_noise_schedule(TIMESTEPS, device=device)

MODEL_TYPE = 'BackboneCoordinateEGNNDenoiser'
MODEL_HIDDEN_DIM = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_HIDDEN_DIM', '192'))
MODEL_NUM_LAYERS = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_NUM_LAYERS', '4'))
MODEL_TIME_EMBEDDING_DIM = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_TIME_EMBEDDING_DIM', '128'))
MODEL_SEQUENCE_OFFSET_EDGES = tuple(int(value) for value in os.environ.get('BACKBONE_DIFFUSION_SEQUENCE_OFFSET_EDGES', '8,16,32').split(',') if value.strip())
model = BackboneCoordinateEGNNDenoiser(
    max_length=MAX_SEQ_LENGTH,
    hidden_dim=MODEL_HIDDEN_DIM,
    num_layers=MODEL_NUM_LAYERS,
    time_embedding_dim=MODEL_TIME_EMBEDDING_DIM,
    sequence_offset_edges=MODEL_SEQUENCE_OFFSET_EDGES,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)
print('Model type:', MODEL_TYPE)
print('Model parameters:', sum(parameter.numel() for parameter in model.parameters()))
print('Sequence-offset CA edges:', MODEL_SEQUENCE_OFFSET_EDGES)
print('Schedule keys:', list(noise_schedule.keys()))
print('Schedule shapes:')
for key, value in noise_schedule.items():
    print(key, tuple(value.shape), value.device)

with torch.no_grad():
    demo_coords = coords_norm.to(device)
    demo_mask = mask.to(device)
    demo_t = sample_timesteps(demo_coords.shape[0], TIMESTEPS, device)
    demo_noise = torch.randn_like(flatten_backbone(demo_coords))
    demo_x0 = flatten_backbone(demo_coords)
    demo_xt = q_sample(demo_x0, demo_t, demo_noise, noise_schedule['alpha_bars'])
    demo_pred = model(demo_xt, demo_t, demo_mask)
    demo_noise_loss = masked_noise_mse(demo_pred, demo_noise, demo_mask)

print('x0 shape:', tuple(demo_x0.shape))
print('xt shape:', tuple(demo_xt.shape))
print('pred shape:', tuple(demo_pred.shape))
print('demo masked noise loss:', float(demo_noise_loss.detach().cpu()))
assert demo_pred.shape == demo_x0.shape


## 7. Training, validation, and test evaluation

The loop tracks total objective, normal DDPM denoising loss, bond geometry loss, adjacent-CA geometry loss, and x0 reconstruction RMSE. Geometry losses are computed only after converting `x0_pred` back from normalized flattened coordinates into `B x L x 4 x 3` Angstrom coordinates.

V4 inherits the stable V3b objective directly: Smooth L1 geometry losses, `lambda_bond = 0.01`, `lambda_ca = 0.01`, geometry terms active only for timesteps `t <= 50`, and no radius anti-collapse term. The architecture change is the point of the run, not a new loss tweak.


In [ ]:
def prepare_batch_for_diffusion(batch: dict[str, object], target_device: torch.device, stats: BackboneNormalizationStats) -> tuple[torch.Tensor, torch.Tensor]:
    """Move a batch to device and return normalized flattened coords plus the mask."""
    batch = move_batch_to_device(batch, target_device)
    coords = batch['coords']
    mask = batch['mask']
    coords_centered = centre_coordinates(coords, mask)
    coords_norm = apply_coordinate_normalisation(coords_centered, mask, stats)
    coords_flat = flatten_backbone(coords_norm)
    return coords_flat, mask

NOISE_LOSS_WEIGHT = 1.0
LAMBDA_BOND = float(os.environ.get('BACKBONE_DIFFUSION_LAMBDA_BOND', '0.01'))
LAMBDA_CA = float(os.environ.get('BACKBONE_DIFFUSION_LAMBDA_CA', '0.01'))
GEOMETRY_LOSS_BETA = float(os.environ.get('BACKBONE_DIFFUSION_GEOMETRY_LOSS_BETA', '0.5'))
GEOMETRY_MAX_TIMESTEP = int(os.environ.get('BACKBONE_DIFFUSION_GEOMETRY_MAX_TIMESTEP', '50'))
GEOMETRY_TARGETS_ANGSTROM = {
    'n_ca': 1.46,
    'ca_c': 1.53,
    'c_o': 1.23,
    'c_n': 1.33,
    'adjacent_ca': 3.80,
}

def geometry_timestep_mask(t: torch.Tensor, target_ndim: int) -> torch.Tensor:
    """Return a broadcastable mask that enables geometry loss only for selected timesteps."""
    active = (t <= GEOMETRY_MAX_TIMESTEP).float()
    view_shape = (t.shape[0],) + (1,) * (target_ndim - 1)
    return active.view(view_shape)

def masked_distance_huber(distances: torch.Tensor, target: float, valid_mask: torch.Tensor) -> torch.Tensor:
    """Smooth L1 distance error over valid bond or adjacent-residue positions."""
    valid = valid_mask.to(device=distances.device, dtype=distances.dtype)
    target_distances = torch.full_like(distances, fill_value=target)
    per_distance_loss = torch.nn.functional.smooth_l1_loss(
        distances,
        target_distances,
        beta=GEOMETRY_LOSS_BETA,
        reduction='none',
    )
    denom = valid.sum().clamp_min(1.0)
    return (per_distance_loss * valid).sum() / denom

def differentiable_predict_x0(
    x_t: torch.Tensor,
    t: torch.Tensor,
    pred_noise: torch.Tensor,
    alpha_bars: torch.Tensor,
) -> torch.Tensor:
    """Reconstruct x0 while preserving gradients for geometry losses."""
    alpha_bar_t = alpha_bars[t].view(-1, 1, 1).to(device=x_t.device, dtype=x_t.dtype)
    return (x_t - torch.sqrt(1.0 - alpha_bar_t) * pred_noise) / torch.sqrt(alpha_bar_t)

def x0_pred_to_angstrom_coords(x0_pred: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Convert normalized flattened x0 prediction to B x L x 4 x 3 Angstrom coordinates."""
    return invert_coordinate_normalisation(unflatten_backbone(x0_pred), stats)

def adjacent_ca_geometry_loss(x0_pred: torch.Tensor, mask: torch.Tensor, t: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Penalize adjacent CA distances at timesteps where x0 predictions are stable enough."""
    coords_angstrom = x0_pred_to_angstrom_coords(x0_pred, stats)
    ca = coords_angstrom[:, :, 1, :]
    adjacent_mask = mask[:, :-1].bool() & mask[:, 1:].bool()
    adjacent_mask = adjacent_mask.float() * geometry_timestep_mask(t, target_ndim=2).to(device=mask.device)
    adjacent_ca = torch.linalg.norm(ca[:, 1:, :] - ca[:, :-1, :], dim=-1)
    return masked_distance_huber(adjacent_ca, GEOMETRY_TARGETS_ANGSTROM['adjacent_ca'], adjacent_mask)

def backbone_bond_geometry_loss(x0_pred: torch.Tensor, mask: torch.Tensor, t: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Penalize backbone bond lengths at timesteps where x0 predictions are stable enough."""
    coords_angstrom = x0_pred_to_angstrom_coords(x0_pred, stats)
    timestep_mask = geometry_timestep_mask(t, target_ndim=2).to(device=mask.device)
    residue_mask = mask.bool().float() * timestep_mask
    adjacent_mask = (mask[:, :-1].bool() & mask[:, 1:].bool()).float() * timestep_mask

    n_ca = torch.linalg.norm(coords_angstrom[:, :, 0, :] - coords_angstrom[:, :, 1, :], dim=-1)
    ca_c = torch.linalg.norm(coords_angstrom[:, :, 1, :] - coords_angstrom[:, :, 2, :], dim=-1)
    c_o = torch.linalg.norm(coords_angstrom[:, :, 2, :] - coords_angstrom[:, :, 3, :], dim=-1)
    c_n = torch.linalg.norm(coords_angstrom[:, :-1, 2, :] - coords_angstrom[:, 1:, 0, :], dim=-1)

    components = torch.stack([
        masked_distance_huber(n_ca, GEOMETRY_TARGETS_ANGSTROM['n_ca'], residue_mask),
        masked_distance_huber(ca_c, GEOMETRY_TARGETS_ANGSTROM['ca_c'], residue_mask),
        masked_distance_huber(c_o, GEOMETRY_TARGETS_ANGSTROM['c_o'], residue_mask),
        masked_distance_huber(c_n, GEOMETRY_TARGETS_ANGSTROM['c_n'], adjacent_mask),
    ])
    return components.mean()

def run_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None,
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
) -> dict[str, float]:
    """Run one training or evaluation epoch."""
    is_training = optimizer is not None
    model.train(is_training)
    totals = defaultdict(float)
    n_batches = 0

    for batch in loader:
        coords_flat, mask = prepare_batch_for_diffusion(batch, target_device, stats)
        t = sample_timesteps(coords_flat.shape[0], TIMESTEPS, target_device)
        noise = torch.randn_like(coords_flat)
        x_t = q_sample(coords_flat, t, noise, schedule['alpha_bars'])

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            pred_noise = model(x_t, t, mask)
            noise_loss = masked_noise_mse(pred_noise, noise, mask)
            x0_pred = differentiable_predict_x0(x_t, t, pred_noise, schedule['alpha_bars'])
            bond_loss = backbone_bond_geometry_loss(x0_pred, mask, t, stats)
            ca_loss = adjacent_ca_geometry_loss(x0_pred, mask, t, stats)
            total_loss = (NOISE_LOSS_WEIGHT * noise_loss) + (LAMBDA_BOND * bond_loss) + (LAMBDA_CA * ca_loss)
            recon_rmse = masked_coordinate_rmse(x0_pred, coords_flat, mask)
            if is_training:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        totals['total_loss'] += float(total_loss.detach().cpu())
        totals['noise_loss'] += float(noise_loss.detach().cpu())
        totals['bond_geometry_loss'] += float(bond_loss.detach().cpu())
        totals['adjacent_ca_geometry_loss'] += float(ca_loss.detach().cpu())
        totals['x0_rmse'] += float(recon_rmse.detach().cpu())
        n_batches += 1

    return {key: value / max(n_batches, 1) for key, value in totals.items()}

EPOCHS = int(os.environ.get('BACKBONE_DIFFUSION_EPOCHS', '100'))
history: list[dict[str, float]] = []
best_val_total_loss = float('inf')

print(
    f'Geometry loss settings: bond={LAMBDA_BOND}, adjacent_ca={LAMBDA_CA}, '
    f'beta={GEOMETRY_LOSS_BETA}, active_t<= {GEOMETRY_MAX_TIMESTEP}, radius=0.0'
)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_metrics = run_epoch(model, train_loader, optimizer, normalization_stats, noise_schedule, device)
    val_metrics = run_epoch(model, validation_loader, None, normalization_stats, noise_schedule, device)
    test_metrics = run_epoch(model, test_loader, None, normalization_stats, noise_schedule, device)

    row = {
        'epoch': epoch,
        'train_total_loss': train_metrics['total_loss'],
        'train_noise_loss': train_metrics['noise_loss'],
        'train_bond_geometry_loss': train_metrics['bond_geometry_loss'],
        'train_adjacent_ca_geometry_loss': train_metrics['adjacent_ca_geometry_loss'],
        'train_x0_rmse': train_metrics['x0_rmse'],
        'validation_total_loss': val_metrics['total_loss'],
        'validation_noise_loss': val_metrics['noise_loss'],
        'validation_bond_geometry_loss': val_metrics['bond_geometry_loss'],
        'validation_adjacent_ca_geometry_loss': val_metrics['adjacent_ca_geometry_loss'],
        'validation_x0_rmse': val_metrics['x0_rmse'],
        'test_total_loss': test_metrics['total_loss'],
        'test_noise_loss': test_metrics['noise_loss'],
        'test_bond_geometry_loss': test_metrics['bond_geometry_loss'],
        'test_adjacent_ca_geometry_loss': test_metrics['adjacent_ca_geometry_loss'],
        'test_x0_rmse': test_metrics['x0_rmse'],
        'lr': optimizer.param_groups[0]['lr'],
        'epoch_seconds': time.time() - start_time,
    }
    history.append(row)
    if row['validation_total_loss'] < best_val_total_loss:
        best_val_total_loss = row['validation_total_loss']
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'normalization_mean': normalization_stats.mean,
                'normalization_std': normalization_stats.std,
                'timesteps': TIMESTEPS,
                'max_length': MAX_SEQ_LENGTH,
                'batch_size': BATCH_SIZE,
                'seed': SEED,
                'model_type': MODEL_TYPE,
                'noise_loss_weight': NOISE_LOSS_WEIGHT,
                'lambda_bond': LAMBDA_BOND,
                'lambda_ca': LAMBDA_CA,
                'geometry_loss_beta': GEOMETRY_LOSS_BETA,
                'geometry_max_timestep': GEOMETRY_MAX_TIMESTEP,
                'geometry_loss_type': 'smooth_l1',
                'geometry_targets_angstrom': GEOMETRY_TARGETS_ANGSTROM,
                'model_hidden_dim': MODEL_HIDDEN_DIM,
                'model_num_layers': MODEL_NUM_LAYERS,
                'model_time_embedding_dim': MODEL_TIME_EMBEDDING_DIM,
                'model_sequence_offset_edges': MODEL_SEQUENCE_OFFSET_EDGES,
                'saved_epoch': epoch,
                'saved_validation_total_loss': row['validation_total_loss'],
                'saved_validation_noise_loss': row['validation_noise_loss'],
            },
            CHECKPOINT_DIR / 'best_backbone_diffusion.pt',
        )
    print(
        f"Epoch {epoch:02d} | train total {row['train_total_loss']:.5f} | val total {row['validation_total_loss']:.5f} | "
        f"val noise {row['validation_noise_loss']:.5f} | val bond {row['validation_bond_geometry_loss']:.4f} | "
        f"val CA {row['validation_adjacent_ca_geometry_loss']:.4f} | {row['epoch_seconds']:.1f}s"
    )

history_df = pd.DataFrame(history)
display(history_df)
save_table_artifact(history_df, 'backbone_diffusion_history')



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df['epoch'], history_df['train_total_loss'], label='Train total')
axes[0].plot(history_df['epoch'], history_df['validation_total_loss'], label='Validation total')
axes[0].plot(history_df['epoch'], history_df['test_total_loss'], label='Test total')
axes[0].plot(history_df['epoch'], history_df['validation_noise_loss'], linestyle='--', label='Validation noise')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('V4 training objective')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['validation_bond_geometry_loss'], marker='o', label='Validation bond geometry')
axes[1].plot(history_df['epoch'], history_df['validation_adjacent_ca_geometry_loss'], marker='o', label='Validation adjacent CA')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Unweighted Smooth L1 geometry loss')
axes[1].set_title('V4 geometry losses')
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'backbone_diffusion_losses.png', dpi=150)
plt.show()


## 8. Sampling and evaluation

This section reloads the saved best-validation checkpoint before sampling, then denormalizes sampled backbones back into coordinate space for structural evaluation and visualization. That keeps the reported qualitative and quantitative outputs aligned with the model-selection rule used during training.


In [ ]:
checkpoint_path = CHECKPOINT_DIR / 'best_backbone_diffusion.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(
    f"Loaded best checkpoint from {checkpoint_path} | epoch={checkpoint.get('saved_epoch', 'unknown')} | "
    f"val_total={checkpoint.get('saved_validation_total_loss', 'unknown')} | "
    f"val_noise={checkpoint.get('saved_validation_noise_loss', 'unknown')}"
)

model.eval()
with torch.no_grad():
    eval_batch = next(iter(validation_loader))
    sample_mask = eval_batch['mask'][:4].to(device)
    sampled_flat = sample_backbone(
        model,
        noise_schedule,
        shape=(sample_mask.shape[0], MAX_SEQ_LENGTH, 12),
        mask=sample_mask,
        device=device,
    )
    sampled_coords_norm = unflatten_backbone(sampled_flat)
    sampled_coords = invert_coordinate_normalisation(sampled_coords_norm, normalization_stats)

    real_coords = eval_batch['coords'][:4]
    real_mask = eval_batch['mask'][:4]

real_rows = []
generated_rows = []
for index in range(sampled_coords.shape[0]):
    real_rows.append({'kind': 'real', 'index': index, **backbone_structure_summary(real_coords[index], real_mask[index])})
    generated_rows.append({'kind': 'generated', 'index': index, **backbone_structure_summary(sampled_coords[index].cpu(), sample_mask[index].cpu())})

real_eval_df = pd.DataFrame(real_rows)
generated_eval_df = pd.DataFrame(generated_rows)

display(real_eval_df)
display(generated_eval_df)

comparison_df = pd.concat([real_eval_df, generated_eval_df], ignore_index=True)
comparison_summary = comparison_df.groupby('kind', as_index=False).mean(numeric_only=True)
display(comparison_summary)
save_table_artifact(real_eval_df, 'real_eval_metrics')
save_table_artifact(generated_eval_df, 'generated_eval_metrics')
save_table_artifact(comparison_df, 'real_vs_generated_summary')
save_table_artifact(comparison_summary, 'real_vs_generated_summary_mean')


In [ ]:
for index in range(min(2, sampled_coords.shape[0])):
    print(f'Generated example {index}')
    print(backbone_structure_summary(sampled_coords[index].cpu(), sample_mask[index].cpu()))
    plot_ca_trace_from_batch(sampled_coords[index].cpu(), sample_mask[index].bool().cpu(), title=f'Generated CA trace {index}')
    render_backbone_example(sampled_coords[index].cpu(), sample_mask[index].cpu(), title=f'Generated backbone {index}')


## 9. V4 diagnostic evaluation

V4 keeps the V2/V3/V3b diagnostic harness, but evaluates a coordinate-aware EGNN-style denoiser under the same stable V3b geometry objective. The key question is whether sparse equivariant-style coordinate updates can preserve the local geometry gains while reducing global collapse.


In [ ]:
model_parameter_count = sum(parameter.numel() for parameter in model.parameters())
train_chunk_count_summary = train_chunked_df.groupby('parent_chain_id').size()
best_epoch_row = history_df.loc[history_df['validation_total_loss'].idxmin()].to_dict()

def config_row(key: str, value: object) -> dict[str, object]:
    """Return a Parquet-safe config row with stable column types."""
    is_bool = isinstance(value, bool)
    is_int = isinstance(value, int) and not is_bool
    is_float = isinstance(value, float)
    return {
        'key': key,
        'value_text': str(value),
        'value_int': int(value) if is_int else pd.NA,
        'value_float': float(value) if (is_int or is_float) else np.nan,
    }

run_config_summary = pd.DataFrame([
    config_row('artifact_run_name', ARTIFACT_RUN_NAME),
    config_row('seed', SEED),
    config_row('device', str(device)),
    config_row('device_name', device_name),
    config_row('max_seq_length', MAX_SEQ_LENGTH),
    config_row('backbone_atoms', ','.join(BACKBONE_ATOMS)),
    config_row('train_chains', len(train_df)),
    config_row('validation_chains', len(validation_df)),
    config_row('test_chains', len(test_df)),
    config_row('train_chunks', len(train_chunked_df)),
    config_row('train_chunk_stride', TRAIN_CHUNK_STRIDE),
    config_row('median_chunks_per_train_chain', float(train_chunk_count_summary.median())),
    config_row('max_chunks_per_train_chain', int(train_chunk_count_summary.max())),
    config_row('timesteps', TIMESTEPS),
    config_row('epochs', EPOCHS),
    config_row('batch_size', BATCH_SIZE),
    config_row('optimizer', 'Adam'),
    config_row('learning_rate', optimizer.param_groups[0]['lr']),
    config_row('model_type', MODEL_TYPE),
    config_row('model_hidden_dim', MODEL_HIDDEN_DIM),
    config_row('model_num_layers', MODEL_NUM_LAYERS),
    config_row('model_time_embedding_dim', MODEL_TIME_EMBEDDING_DIM),
    config_row('model_sequence_offset_edges', ','.join(str(value) for value in MODEL_SEQUENCE_OFFSET_EDGES)),
    config_row('model_parameter_count', model_parameter_count),
    config_row('noise_loss_weight', NOISE_LOSS_WEIGHT),
    config_row('lambda_bond', LAMBDA_BOND),
    config_row('lambda_ca', LAMBDA_CA),
    config_row('lambda_radius_of_gyration', 0.0),
    config_row('geometry_loss_type', 'smooth_l1'),
    config_row('geometry_loss_beta', GEOMETRY_LOSS_BETA),
    config_row('geometry_max_timestep', GEOMETRY_MAX_TIMESTEP),
    config_row('target_n_ca_angstrom', GEOMETRY_TARGETS_ANGSTROM['n_ca']),
    config_row('target_ca_c_angstrom', GEOMETRY_TARGETS_ANGSTROM['ca_c']),
    config_row('target_c_o_angstrom', GEOMETRY_TARGETS_ANGSTROM['c_o']),
    config_row('target_c_n_angstrom', GEOMETRY_TARGETS_ANGSTROM['c_n']),
    config_row('target_adjacent_ca_angstrom', GEOMETRY_TARGETS_ANGSTROM['adjacent_ca']),
    config_row('best_epoch', int(best_epoch_row['epoch'])),
    config_row('best_validation_total_loss', float(best_epoch_row['validation_total_loss'])),
    config_row('best_validation_noise_loss', float(best_epoch_row['validation_noise_loss'])),
    config_row('best_test_total_loss_at_reported_epoch', float(best_epoch_row['test_total_loss'])),
    config_row('best_test_noise_loss_at_reported_epoch', float(best_epoch_row['test_noise_loss'])),
])
run_config_summary['value_int'] = run_config_summary['value_int'].astype('Int64')

display(run_config_summary)
save_table_artifact(run_config_summary, 'v4_run_config_summary')



### Fixed-timestep denoising diagnostics

The training loss samples timesteps uniformly, which is good for optimization but hides where the model is weak. This diagnostic evaluates the same geometry-augmented trained model at fixed low, medium, and high noise levels on held-out data.


In [ ]:
@torch.no_grad()
def evaluate_fixed_timesteps(
    model: torch.nn.Module,
    loader: DataLoader,
    split_name: str,
    timesteps_to_check: list[int],
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
    max_batches: int | None = None,
) -> pd.DataFrame:
    model.eval()
    rows = []
    for timestep in timesteps_to_check:
        totals = defaultdict(float)
        n_batches = 0
        for batch_index, batch in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            coords_flat, mask = prepare_batch_for_diffusion(batch, target_device, stats)
            t = torch.full((coords_flat.shape[0],), timestep, device=target_device, dtype=torch.long)
            noise = torch.randn_like(coords_flat)
            x_t = q_sample(coords_flat, t, noise, schedule['alpha_bars'])
            pred_noise = model(x_t, t, mask)
            noise_loss = masked_noise_mse(pred_noise, noise, mask)
            x0_pred = differentiable_predict_x0(x_t, t, pred_noise, schedule['alpha_bars'])
            bond_loss = backbone_bond_geometry_loss(x0_pred, mask, t, stats)
            ca_loss = adjacent_ca_geometry_loss(x0_pred, mask, t, stats)
            total_loss = (NOISE_LOSS_WEIGHT * noise_loss) + (LAMBDA_BOND * bond_loss) + (LAMBDA_CA * ca_loss)
            recon_rmse = masked_coordinate_rmse(x0_pred, coords_flat, mask)
            totals['total_loss'] += float(total_loss.detach().cpu())
            totals['noise_loss'] += float(noise_loss.detach().cpu())
            totals['bond_geometry_loss'] += float(bond_loss.detach().cpu())
            totals['adjacent_ca_geometry_loss'] += float(ca_loss.detach().cpu())
            totals['x0_rmse'] += float(recon_rmse.detach().cpu())
            n_batches += 1
        rows.append({
            'split': split_name,
            'timestep': timestep,
            'noise_fraction': timestep / TIMESTEPS,
            'total_loss': totals['total_loss'] / max(n_batches, 1),
            'noise_loss': totals['noise_loss'] / max(n_batches, 1),
            'bond_geometry_loss': totals['bond_geometry_loss'] / max(n_batches, 1),
            'adjacent_ca_geometry_loss': totals['adjacent_ca_geometry_loss'] / max(n_batches, 1),
            'x0_rmse': totals['x0_rmse'] / max(n_batches, 1),
            'n_batches': n_batches,
        })
    return pd.DataFrame(rows)

TIMESTEP_DIAGNOSTIC_POINTS = [1, 5, 10, 25, 50, 75, 100]
validation_timestep_df = evaluate_fixed_timesteps(
    model,
    validation_loader,
    'validation',
    TIMESTEP_DIAGNOSTIC_POINTS,
    normalization_stats,
    noise_schedule,
    device,
)
test_timestep_df = evaluate_fixed_timesteps(
    model,
    test_loader,
    'test',
    TIMESTEP_DIAGNOSTIC_POINTS,
    normalization_stats,
    noise_schedule,
    device,
)
timestep_diagnostics_df = pd.concat([validation_timestep_df, test_timestep_df], ignore_index=True)

display(timestep_diagnostics_df)
save_table_artifact(timestep_diagnostics_df, 'v4_timestep_diagnostics')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split_name, split_df in timestep_diagnostics_df.groupby('split'):
    axes[0].plot(split_df['timestep'], split_df['noise_loss'], marker='o', label=f'{split_name} noise')
    axes[1].plot(split_df['timestep'], split_df['bond_geometry_loss'], marker='o', label=f'{split_name} bond')
    axes[1].plot(split_df['timestep'], split_df['adjacent_ca_geometry_loss'], marker='s', linestyle='--', label=f'{split_name} CA')
axes[0].set_xlabel('Diffusion timestep')
axes[0].set_ylabel('Masked noise MSE')
axes[0].set_title('V4 fixed-timestep denoising')
axes[0].legend()
axes[1].set_xlabel('Diffusion timestep')
axes[1].set_ylabel('Smooth L1 geometry loss')
axes[1].set_title('V4 fixed-timestep geometry')
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4_timestep_diagnostics.png', dpi=150)
plt.show()


### Broader sample-quality diagnostics

V4 samples the same number of validation-shaped masks as V2/V3/V3b and computes the same structural summaries. When reference artifact tables are present, the notebook writes direct metric and collapse comparisons against V2, V3, V3b, and optionally V3c.


In [ ]:
@torch.no_grad()
def collect_conditioning_masks(loader: DataLoader, target_count: int) -> tuple[torch.Tensor, torch.Tensor]:
    coords_batches = []
    mask_batches = []
    seen = 0
    for batch in loader:
        coords = batch['coords']
        mask = batch['mask']
        remaining = target_count - seen
        coords_batches.append(coords[:remaining])
        mask_batches.append(mask[:remaining])
        seen += min(coords.shape[0], remaining)
        if seen >= target_count:
            break
    return torch.cat(coords_batches, dim=0), torch.cat(mask_batches, dim=0)

@torch.no_grad()
def sample_and_evaluate_masks(
    model: torch.nn.Module,
    masks: torch.Tensor,
    real_coords: torch.Tensor,
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
    sample_batch_size: int = 8,
) -> tuple[pd.DataFrame, pd.DataFrame, torch.Tensor]:
    model.eval()
    generated_coord_batches = []
    real_rows = []
    generated_rows = []
    for start in range(0, masks.shape[0], sample_batch_size):
        end = min(start + sample_batch_size, masks.shape[0])
        batch_mask = masks[start:end].to(target_device)
        sampled_flat = sample_backbone(
            model,
            schedule,
            shape=(batch_mask.shape[0], MAX_SEQ_LENGTH, 12),
            mask=batch_mask,
            device=target_device,
        )
        sampled_coords_norm = unflatten_backbone(sampled_flat)
        generated_coords = invert_coordinate_normalisation(sampled_coords_norm, stats).cpu()
        generated_coord_batches.append(generated_coords)
        for local_index in range(generated_coords.shape[0]):
            index = start + local_index
            real_rows.append({
                'kind': 'real',
                'index': index,
                **backbone_structure_summary(real_coords[index], masks[index]),
            })
            generated_rows.append({
                'kind': 'generated',
                'index': index,
                **backbone_structure_summary(generated_coords[local_index], masks[index]),
            })
    return pd.DataFrame(real_rows), pd.DataFrame(generated_rows), torch.cat(generated_coord_batches, dim=0)

def build_metric_summary(comparison_df: pd.DataFrame) -> pd.DataFrame:
    return comparison_df.groupby('kind', as_index=False).agg(
        n_structures=('index', 'count'),
        mean_n_residues=('n_residues', 'mean'),
        mean_adjacent_ca=('mean_adjacent_ca', 'mean'),
        std_adjacent_ca=('mean_adjacent_ca', 'std'),
        mean_fraction_adjacent_ca_in_band=('fraction_adjacent_ca_in_band', 'mean'),
        mean_n_ca=('mean_n_ca', 'mean'),
        mean_ca_c=('mean_ca_c', 'mean'),
        mean_c_o=('mean_c_o', 'mean'),
        mean_c_n=('mean_c_n', 'mean'),
        mean_radius_of_gyration=('radius_of_gyration', 'mean'),
        std_radius_of_gyration=('radius_of_gyration', 'std'),
    )

def build_structural_gap_summary(metric_summary: pd.DataFrame) -> pd.DataFrame:
    real_means = metric_summary[metric_summary['kind'] == 'real'].iloc[0]
    generated_means = metric_summary[metric_summary['kind'] == 'generated'].iloc[0]
    return pd.DataFrame([
        {
            'metric': 'mean_adjacent_ca',
            'real_mean': real_means['mean_adjacent_ca'],
            'generated_mean': generated_means['mean_adjacent_ca'],
            'generated_minus_real': generated_means['mean_adjacent_ca'] - real_means['mean_adjacent_ca'],
            'interpretation': 'Generated adjacent CA spacing should be close to the real mean around 3.8 A.',
        },
        {
            'metric': 'fraction_adjacent_ca_in_band',
            'real_mean': real_means['mean_fraction_adjacent_ca_in_band'],
            'generated_mean': generated_means['mean_fraction_adjacent_ca_in_band'],
            'generated_minus_real': generated_means['mean_fraction_adjacent_ca_in_band'] - real_means['mean_fraction_adjacent_ca_in_band'],
            'interpretation': 'Low generated fraction indicates broken local CA geometry.',
        },
        {
            'metric': 'radius_of_gyration',
            'real_mean': real_means['mean_radius_of_gyration'],
            'generated_mean': generated_means['mean_radius_of_gyration'],
            'generated_minus_real': generated_means['mean_radius_of_gyration'] - real_means['mean_radius_of_gyration'],
            'interpretation': 'Low generated radius indicates global collapse or over-compact sampling.',
        },
    ])

def adjacent_ca_distribution_summary(eval_df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Summarize per-structure adjacent CA metrics beyond the mean."""
    frame = eval_df.copy()
    return pd.DataFrame([
        {
            'kind': label,
            'n_structures': len(frame),
            'mean_adjacent_ca_mean': frame['mean_adjacent_ca'].mean(),
            'mean_adjacent_ca_std': frame['mean_adjacent_ca'].std(),
            'mean_adjacent_ca_p05': frame['mean_adjacent_ca'].quantile(0.05),
            'mean_adjacent_ca_p25': frame['mean_adjacent_ca'].quantile(0.25),
            'mean_adjacent_ca_median': frame['mean_adjacent_ca'].median(),
            'mean_adjacent_ca_p75': frame['mean_adjacent_ca'].quantile(0.75),
            'mean_adjacent_ca_p95': frame['mean_adjacent_ca'].quantile(0.95),
            'ca_in_band_mean': frame['fraction_adjacent_ca_in_band'].mean(),
            'ca_in_band_p25': frame['fraction_adjacent_ca_in_band'].quantile(0.25),
            'ca_in_band_median': frame['fraction_adjacent_ca_in_band'].median(),
            'ca_in_band_p75': frame['fraction_adjacent_ca_in_band'].quantile(0.75),
        }
    ])

def load_reference_metric_summary(version: str) -> pd.DataFrame | None:
    reference_path = ARTIFACT_BASE_DIR / version / 'tables' / f'{version}_real_vs_generated_metric_summary.csv'
    if not reference_path.exists():
        print(f'{version.upper()} reference metric summary not found at {reference_path}; skipping direct metric comparison.')
        return None
    return pd.read_csv(reference_path)

def load_reference_collapse_summary(version: str) -> pd.DataFrame | None:
    reference_path = ARTIFACT_BASE_DIR / version / 'tables' / f'{version}_collapse_summary.csv'
    if not reference_path.exists():
        print(f'{version.upper()} reference collapse summary not found at {reference_path}; skipping collapse comparison.')
        return None
    return pd.read_csv(reference_path)

def build_metric_comparison(reference_version: str, reference_summary: pd.DataFrame, current_summary: pd.DataFrame) -> pd.DataFrame:
    reference_generated = reference_summary[reference_summary['kind'] == 'generated'].iloc[0]
    current_generated = current_summary[current_summary['kind'] == 'generated'].iloc[0]
    current_real = current_summary[current_summary['kind'] == 'real'].iloc[0]
    metrics = [
        ('mean_adjacent_ca', 'increase toward 3.8 A'),
        ('mean_fraction_adjacent_ca_in_band', 'increase'),
        ('mean_radius_of_gyration', 'increase toward real'),
        ('mean_n_ca', 'move toward real'),
        ('mean_ca_c', 'move toward real'),
        ('mean_c_o', 'move toward real'),
        ('mean_c_n', 'move toward real'),
    ]
    rows = []
    for metric, target_direction in metrics:
        rows.append({
            'metric': metric,
            'real_mean': current_real[metric],
            f'{reference_version}_generated_mean': reference_generated[metric],
            'v4_generated_mean': current_generated[metric],
            f'v4_minus_{reference_version}': current_generated[metric] - reference_generated[metric],
            'target_direction': target_direction,
        })
    return pd.DataFrame(rows)

def build_collapse_comparison(reference_version: str, reference_collapse: pd.DataFrame, current_collapse: pd.DataFrame) -> pd.DataFrame:
    reference = reference_collapse.iloc[0]
    current = current_collapse.iloc[0]
    reference_sample_count = int(reference['sample_count'])
    current_sample_count = int(current['sample_count'])
    reference_collapse_count = int(round(float(reference['collapse_fraction']) * reference_sample_count))
    reference_poor_ca_count = int(round(float(reference['poor_ca_band_fraction']) * reference_sample_count))
    current_collapse_count = int(current.get('collapse_count', round(float(current['collapse_fraction']) * current_sample_count)))
    current_poor_ca_count = int(current.get('poor_ca_band_count', round(float(current['poor_ca_band_fraction']) * current_sample_count)))
    return pd.DataFrame([
        {
            'version': reference_version,
            'sample_count': reference_sample_count,
            'collapse_fraction': float(reference['collapse_fraction']),
            'collapse_count': reference_collapse_count,
            'poor_ca_band_fraction': float(reference['poor_ca_band_fraction']),
            'poor_ca_band_count': reference_poor_ca_count,
        },
        {
            'version': 'v4',
            'sample_count': current_sample_count,
            'collapse_fraction': float(current['collapse_fraction']),
            'collapse_count': current_collapse_count,
            'poor_ca_band_fraction': float(current['poor_ca_band_fraction']),
            'poor_ca_band_count': current_poor_ca_count,
        },
    ])

V4_SAMPLE_COUNT = min(32, len(validation_dataset))
v4_real_coords, v4_sample_masks = collect_conditioning_masks(validation_loader, V4_SAMPLE_COUNT)
v4_real_eval_df, v4_generated_eval_df, v4_sampled_coords = sample_and_evaluate_masks(
    model,
    v4_sample_masks,
    v4_real_coords,
    normalization_stats,
    noise_schedule,
    device,
    sample_batch_size=8,
)

v4_comparison_df = pd.concat([v4_real_eval_df, v4_generated_eval_df], ignore_index=True)
v4_metric_summary = build_metric_summary(v4_comparison_df)
v4_gap_summary = build_structural_gap_summary(v4_metric_summary)

real_rg_median = float(v4_real_eval_df['radius_of_gyration'].median())
collapse_radius_threshold = 0.5 * real_rg_median
v4_generated_eval_df['collapse_flag'] = v4_generated_eval_df['radius_of_gyration'] < collapse_radius_threshold
v4_generated_eval_df['poor_ca_band_flag'] = v4_generated_eval_df['fraction_adjacent_ca_in_band'] < 0.8
collapse_summary = pd.DataFrame([
    {
        'sample_count': len(v4_generated_eval_df),
        'real_radius_median': real_rg_median,
        'collapse_radius_threshold': collapse_radius_threshold,
        'collapse_fraction': float(v4_generated_eval_df['collapse_flag'].mean()),
        'collapse_count': int(v4_generated_eval_df['collapse_flag'].sum()),
        'poor_ca_band_fraction': float(v4_generated_eval_df['poor_ca_band_flag'].mean()),
        'poor_ca_band_count': int(v4_generated_eval_df['poor_ca_band_flag'].sum()),
    }
])

for frame in [v4_metric_summary, v4_gap_summary, collapse_summary]:
    display(frame)

save_table_artifact(v4_real_eval_df, 'v4_real_eval_metrics')
save_table_artifact(v4_generated_eval_df, 'v4_generated_eval_metrics')
save_table_artifact(v4_comparison_df, 'v4_real_vs_generated_summary')
save_table_artifact(v4_metric_summary, 'v4_real_vs_generated_metric_summary')
save_table_artifact(v4_gap_summary, 'v4_structural_gap_summary')
save_table_artifact(collapse_summary, 'v4_collapse_summary')

v4_ca_distribution_summary = pd.concat([
    adjacent_ca_distribution_summary(v4_real_eval_df, 'real'),
    adjacent_ca_distribution_summary(v4_generated_eval_df, 'generated'),
], ignore_index=True)
display(v4_ca_distribution_summary)
save_table_artifact(v4_ca_distribution_summary, 'v4_adjacent_ca_distribution_summary')

for reference_version in ['v2', 'v3', 'v3b', 'v3c']:
    reference_metric_summary = load_reference_metric_summary(reference_version)
    if reference_metric_summary is not None:
        metric_comparison = build_metric_comparison(reference_version, reference_metric_summary, v4_metric_summary)
        display(metric_comparison)
        save_table_artifact(metric_comparison, f'v4_vs_{reference_version}_metric_comparison')

    reference_collapse_summary = load_reference_collapse_summary(reference_version)
    if reference_collapse_summary is not None:
        collapse_comparison = build_collapse_comparison(reference_version, reference_collapse_summary, collapse_summary)
        display(collapse_comparison)
        save_table_artifact(collapse_comparison, f'v4_vs_{reference_version}_collapse_comparison')




In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics_to_plot = [
    ('mean_adjacent_ca', 'Mean adjacent CA distance'),
    ('fraction_adjacent_ca_in_band', 'Adjacent CA in-band fraction'),
    ('radius_of_gyration', 'Radius of gyration'),
]
for ax, (metric, label) in zip(axes, metrics_to_plot):
    for kind, frame in v4_comparison_df.groupby('kind'):
        ax.hist(frame[metric].dropna(), bins=12, alpha=0.6, label=kind)
    ax.set_title(label)
    ax.set_xlabel(metric)
    ax.set_ylabel('Count')
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4_structural_metric_distributions.png', dpi=150)
plt.show()


## 10. V4 outcome summary

V4 is the next logical architecture step after V3b and V3c. V2 showed that the flattened DDPM can optimize denoising loss but collapses, V3 showed that local atom-graph message passing alone was not enough, V3b showed that explicit local geometry supervision helps substantially, and V3c showed that a simple radius-hinge anti-collapse term does not reliably fix global structure.

V4 keeps the stable V3b objective and changes the denoiser instead:

- EGNN-style coordinate-aware graph denoiser over backbone atoms
- sparse local backbone edges plus optional nonlocal CA sequence-offset edges
- the same Smooth L1 bond and adjacent-CA losses on denormalized `x0_pred`
- `lambda_bond = 0.01`
- `lambda_ca = 0.01`
- geometry terms active only for timesteps `t <= 50`
- no radius-of-gyration anti-collapse loss in the initial V4 run

The intended interpretation is direct:

- if adjacent CA in-band fraction stays near or above the V3b level, the coordinate-aware architecture preserved the local-geometry gains
- if adjacent CA mean moves toward 3.8 A and bond means stay closer to real values than V2, the sparse EGNN bias is helping local structure rather than damaging it
- if radius of gyration rises and collapse count falls below V3b, the architecture improved the known global-collapse failure mode
- if metrics stay flat, the next step should look at stronger nonlocal edges, sampling-time guidance, or more explicit global structure constraints rather than another simple radius hinge

The notebook saves V4 history, checkpoint, run config, fixed-timestep diagnostics, generated metrics, adjacent-CA distribution summaries, structural summaries, collapse flags, plus direct `v4_vs_v2_*`, `v4_vs_v3_*`, `v4_vs_v3b_*`, and optional `v4_vs_v3c_*` comparison tables when reference artifacts are available under the same artifact base directory.
